# MoMa 01 — Tyrosinase and MAO in substantia nigra DA neurons: feasibility

**Project:** MoMa (ASAP-030013) · **Runs in:** Verily Workbench (JupyterLab) · **Data:** ASAP CRN Cloud curated sc/snRNA-seq

**Question for this notebook.** Before any disease analysis, can we actually see what we need?

1. Can we find substantia nigra (SN) dopaminergic (DA) nuclei, and how did the CRN pipeline label them?
2. Is **TYR** detectable at all in DA nuclei (UMIs, fraction of nuclei, per donor)?
3. What do **MAOA / MAOB / COMT / ALDH1A1** look like in DA nuclei (and MAOB in astrocytes)?
4. First, exploratory donor-level look: PD vs control, and TYR/MAO vs a DA-loss proxy.

The notebook processes **one dataset at a time** (set `DATASET` below). Start with `lee`, then `sulzer`.

> **Data handling (award §5).** CRN data and every table/figure derived from it stay inside this workspace until
> released. Outputs are written to the app disk / workspace bucket only — do not download them.

In [ ]:
%pip install anndata

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# make moma_utils importable (repo cloned to $HOME/repos/<name>/ in Workbench)
for p in [Path.cwd(), Path.cwd().parent, *Path.home().glob("repos/*")]:
    if (p / "moma_utils.py").exists():
        sys.path.insert(0, str(p)); break
import moma_utils as mu

pd.set_option("display.max_columns", 50)
print("moma_utils loaded from", mu.__file__)

## 0 · Configuration

Bucket paths follow the CRN Cloud Rosetta Stone pattern
`gs://asap-curated-team-<dataset>/<collection>/release/<curated-in release>`.
**Check them once against the [Dataset Finder](https://asap-crn.github.io/asap-crn-learning-lab/rosetta-stone/datasets/)** — the
"curated in" release can lag behind the latest CRN release.

`MOMA_TEST_H5AD`: if this environment variable points to a local `.h5ad`, the notebook skips GCS entirely.
Use that to develop on public or synthetic data (e.g. in Colab) before running on CRN data here.

In [ ]:
DATASET = "lee"   # "lee" or "sulzer"

DATASETS = {
    # Lee: snRNA-seq of hippocampus, middle frontal gyrus and substantia nigra (curated in v3.0.0)
    "lee":    "gs://asap-curated-team-lee-pmdbs-sn-rnaseq/pmdbs_sc_rnaseq/release/v3.0.0",
    # Sulzer: snRNA-seq of cingulate cortex and substantia nigra (curated in v4.0.0)
    "sulzer": "gs://asap-curated-team-sulzer-pmdbs-sn-rnaseq/pmdbs_sc_rnaseq/release/v4.0.0",
}

LOCAL_DIR = Path.home() / "moma_data" / DATASET       # app disk (inside the workspace)
OUT_DIR   = Path.home() / "moma_outputs" / DATASET
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_H5AD = os.environ.get("MOMA_TEST_H5AD")          # optional: local test file
COUNTS_LAYER = "counts"                                # raw UMIs; pipeline stores them here
GENES = list(mu.GENE_PANEL)
print("Dataset:", DATASET, "| test mode:", bool(TEST_H5AD))

## 1 · Find the curated files

In [ ]:
if TEST_H5AD:
    h5ad_path = Path(TEST_H5AD)
    listing = []
else:
    listing = mu.gcs_ls(DATASETS[DATASET])
    h5ads = mu.find_files(listing, r"\.h5ad$")
    csvs  = mu.find_files(listing, r"\.(csv|tsv)$")
    print(f"{len(listing)} objects; {len(h5ads)} h5ad, {len(csvs)} csv/tsv\n")
    for p in h5ads: print(f"{mu.gcs_size(p):>10}  {p}")
    print()
    for p in csvs[:40]: print(p)

    finals = mu.find_files(h5ads, r"final_adata\.h5ad$")
    assert finals, "No *final_adata.h5ad found — pick a file from the list above and set h5ad_gs manually."
    h5ad_gs = finals[0]
    print("\nUsing:", h5ad_gs)

In [ ]:
if not TEST_H5AD:
    h5ad_path = mu.fetch(h5ad_gs, LOCAL_DIR)   # copies once to the app disk; can take a while
adata = mu.LazyH5AD(h5ad_path)
print(adata)
print("\nobs columns:", list(adata.obs.columns))

## 2 · Identify the metadata columns  ⚠️ CHECK

The column names below are guessed from keywords. **Read the printed values and correct the assignments
in the next cell if needed.** If diagnosis is not in `obs`, it lives in the CDE tables (e.g. `SUBJECT`/`SAMPLE`
CSVs in the bucket listing above) and must be merged on the sample/subject ID.

In [ ]:
obs = adata.obs
guesses = {
    "region":    mu.suggest_columns(obs, ["region", "tissue", "brain"]),
    "donor":     mu.suggest_columns(obs, ["subject", "donor", "individual", "patient"]) or
                 mu.suggest_columns(obs, ["sample"]),
    "diagnosis": mu.suggest_columns(obs, ["diagnosis", "condition", "disease", "dx", "status"]),
    "celltype":  mu.suggest_columns(obs, ["scanvi", "cell_type", "celltype", "subclass", "class"]),
}
for k, cols in guesses.items():
    print(f"\n=== {k}: candidates {cols}")
    mu.show_values(obs, cols[:3])

In [ ]:
first = lambda k: guesses[k][0] if guesses[k] else None

REGION_COL   = first("region")      # e.g. "brain_region"
DONOR_COL    = first("donor")       # donor/subject ID — the unit of analysis
DX_COL       = first("diagnosis")   # PD vs control
CELLTYPE_COL = first("celltype")    # pipeline label, e.g. "C_scANVI"

SN_PATTERN   = r"(?i)substantia|nigra|\bSN\b|midbrain"   # values of REGION_COL that count as SN
PD_LABEL, CTRL_LABEL = None, None   # set after looking at DX values, e.g. "PD", "Control"

print(dict(REGION_COL=REGION_COL, DONOR_COL=DONOR_COL, DX_COL=DX_COL, CELLTYPE_COL=CELLTYPE_COL))
if DX_COL:
    print("\nDiagnosis values:", obs[DX_COL].astype(str).value_counts().to_dict())

In [ ]:
# Guess PD / control labels if not set by hand
if DX_COL and (PD_LABEL is None or CTRL_LABEL is None):
    vals = obs[DX_COL].astype(str).unique()
    PD_LABEL   = PD_LABEL   or next((v for v in vals if "parkinson" in v.lower() or v.upper() in ("PD",)), None)
    CTRL_LABEL = CTRL_LABEL or next((v for v in vals if any(w in v.lower() for w in ("control", "healthy", "no pd", "hc"))), None)
print("PD label:", PD_LABEL, "| control label:", CTRL_LABEL)

## 3 · Subset to substantia nigra and pull counts for the gene panel

In [ ]:
if REGION_COL:
    is_sn = obs[REGION_COL].astype(str).str.contains(SN_PATTERN, regex=True)
else:
    print("No region column — using all nuclei (check that this dataset is SN-only).")
    is_sn = pd.Series(True, index=obs.index)
print(f"SN nuclei: {is_sn.sum():,} of {len(obs):,}")
print("Matched region values:", sorted(obs.loc[is_sn, REGION_COL].astype(str).unique()) if REGION_COL else "—")

rows = np.flatnonzero(is_sn.values)
gene_pos = mu.resolve_genes(adata.var, GENES)
missing = [g for g, p in gene_pos.items() if p is None]
if missing: print("Not found in var:", missing)
present = [g for g, p in gene_pos.items() if p is not None]

M = adata.counts(rows, [gene_pos[g] for g in present], layer=COUNTS_LAYER)
counts = pd.DataFrame(M.toarray(), index=obs.index[rows], columns=present)
sn_obs = obs.iloc[rows].copy()

if "total_counts" in sn_obs:
    total_umi = sn_obs["total_counts"].astype(float)
else:
    total_umi = pd.Series(adata.row_sums(rows, layer=COUNTS_LAYER), index=sn_obs.index)
print("Counts pulled:", counts.shape, "| median total UMI/nucleus:", int(total_umi.median()))

## 4 · Marker-based DA calling vs the pipeline's labels

The pipeline's high-confidence neuron labels come from a cortical reference (SEA-AD) and cover GABAergic and
glutamatergic classes only, so DA neurons are expected to land in odd classes or `Unknown`. Here DA nuclei are
called from markers instead: **TH+ and (DAT+ or VMAT2+) and DBH−**.

In [ ]:
sn_obs["cat_call"] = mu.call_catecholaminergic(counts)
print(sn_obs["cat_call"].value_counts().to_string())

if CELLTYPE_COL:
    xt = pd.crosstab(sn_obs[CELLTYPE_COL].astype(str), sn_obs["cat_call"])
    xt = xt.loc[xt.sum(axis=1).sort_values(ascending=False).index]
    display(xt.head(25))
    da_labels = xt["DA"].sort_values(ascending=False) if "DA" in xt else pd.Series(dtype=int)
    print("\nPipeline labels carrying the marker-defined DA nuclei:")
    print((da_labels / max(da_labels.sum(), 1)).head(8).round(3).to_string())

## 5 · Is TYR detectable in DA nuclei?

In [ ]:
is_da = sn_obs["cat_call"] == "DA"
groups = {"DA": is_da, "other SN nuclei": ~is_da}
rows_out = []
for g in [x for x in ["TYR", "TH", "SLC6A3", "MAOA", "MAOB", "COMT", "ALDH1A1"] if x in counts]:
    for name, m in groups.items():
        c = counts.loc[m, g]
        rows_out.append({"gene": g, "nuclei": name, "n": int(m.sum()),
                         "frac_>=1_UMI": round((c >= 1).mean(), 4),
                         "frac_>=2_UMI": round((c >= 2).mean(), 4),
                         "mean_UMI": round(c.mean(), 4), "max_UMI": int(c.max()) if len(c) else 0})
detect = pd.DataFrame(rows_out)
display(detect)

if "TYR" in counts and is_da.any():
    tyr_da = counts.loc[is_da, "TYR"].clip(upper=5).value_counts().sort_index()
    tyr_da.index = [str(i) if i < 5 else "5+" for i in tyr_da.index]
    print("\nTYR UMIs per DA nucleus:\n", tyr_da.to_string())

## 6 · Donor-level pseudobulk (the unit of analysis is the donor)

- **DA pseudobulk:** summed UMIs over each donor's DA nuclei → CPM, plus fraction of DA nuclei with TYR ≥ 1 UMI.
- **DA-loss proxy:** DA nuclei / all neuronal nuclei per donor (neuronal = RBFOX3+ or SNAP25+). Crude — dissection
  and capture vary — but it is the most direct per-donor readout of nigral DA loss available in these data.
- **Astrocyte MAOB:** MAOB pseudobulk in nuclei the pipeline labels as astrocytes (MAOB is largely glial).

In [ ]:
assert DONOR_COL, "Set DONOR_COL in section 2."
donor = sn_obs[DONOR_COL].astype(str)

da_pb = mu.donor_pseudobulk(counts, total_umi, donor, mask=is_da).add_prefix("DA_")

neuron_markers = [g for g in ["RBFOX3", "SNAP25"] if g in counts]
is_neuron = (counts[neuron_markers] >= 1).any(axis=1) | is_da if neuron_markers else is_da
per_donor = pd.DataFrame({
    "n_sn_nuclei": donor.value_counts(),
    "n_neuronal":  donor[is_neuron].value_counts(),
    "n_DA":        donor[is_da].value_counts(),
}).fillna(0).astype(int)
per_donor["DA_frac_of_neuronal"] = per_donor["n_DA"] / per_donor["n_neuronal"].replace(0, np.nan)

if CELLTYPE_COL:
    is_astro = sn_obs[CELLTYPE_COL].astype(str).str.contains("astro", case=False)
    astro_pb = mu.donor_pseudobulk(counts[["MAOB"]] if "MAOB" in counts else counts.iloc[:, :0],
                                   total_umi, donor, mask=is_astro).add_prefix("astro_")
else:
    astro_pb = pd.DataFrame(index=per_donor.index)

donor_tbl = per_donor.join(da_pb, how="left").join(astro_pb, how="left")
if DX_COL:
    donor_tbl["dx"] = sn_obs.groupby(donor, observed=True)[DX_COL].agg(lambda s: s.astype(str).mode().iat[0])
display(donor_tbl.sort_values("n_DA", ascending=False).head(20))
print(f"Donors: {len(donor_tbl)} | donors with >=20 DA nuclei: {(donor_tbl['n_DA'] >= 20).sum()}")

## 7 · First look: PD vs control, and TYR/MAO vs DA loss  (exploratory)

Donors with fewer than `MIN_DA` DA nuclei are excluded from expression comparisons (pseudobulk too noisy).
p-values are uncorrected and descriptive only.

In [ ]:
from scipy.stats import spearmanr
MIN_DA = 20
ok = donor_tbl["n_DA"] >= MIN_DA
features = [c for c in ["DA_TYR_cpm", "DA_TYR_frac_detected", "DA_MAOA_cpm", "DA_MAOB_cpm",
                        "DA_COMT_cpm", "DA_ALDH1A1_cpm", "astro_MAOB_cpm"] if c in donor_tbl]

if DX_COL and PD_LABEL and CTRL_LABEL:
    comp = [mu.compare_groups(donor_tbl[ok], f, "dx", PD_LABEL, CTRL_LABEL) for f in features]
    comp.append(mu.compare_groups(donor_tbl, "DA_frac_of_neuronal", "dx", PD_LABEL, CTRL_LABEL))
    display(pd.DataFrame(comp).round(4))
else:
    print("Set DX_COL / PD_LABEL / CTRL_LABEL in section 2 for the group comparison.")

corr = []
for f in features:
    d = donor_tbl.loc[ok, [f, "DA_frac_of_neuronal"]].dropna()
    if len(d) >= 5:
        r, p = spearmanr(d[f], d["DA_frac_of_neuronal"])
        corr.append({"feature": f, "n_donors": len(d), "spearman_rho_vs_DA_frac": r, "p": p})
display(pd.DataFrame(corr).round(4))

In [ ]:
plot_feats = [f for f in ["DA_TYR_cpm", "DA_MAOA_cpm", "DA_MAOB_cpm", "astro_MAOB_cpm"] if f in donor_tbl]
if plot_feats:
    fig, axes = plt.subplots(1, len(plot_feats), figsize=(3.2 * len(plot_feats), 3.4), squeeze=False)
    d = donor_tbl[ok]
    for ax, f in zip(axes[0], plot_feats):
        if "dx" in d and PD_LABEL and CTRL_LABEL:
            for i, lab in enumerate([CTRL_LABEL, PD_LABEL]):
                y = d.loc[d["dx"] == lab, f].dropna()
                ax.scatter(np.full(len(y), i) + np.random.uniform(-0.12, 0.12, len(y)), y, s=18, alpha=0.8)
                ax.hlines(y.median(), i - 0.25, i + 0.25, color="k", lw=1.5)
            ax.set_xticks([0, 1], [CTRL_LABEL, PD_LABEL])
        else:
            ax.scatter(d["DA_frac_of_neuronal"], d[f], s=18)
            ax.set_xlabel("DA / neuronal nuclei")
        ax.set_title(f, fontsize=9); ax.set_ylabel("CPM (donor pseudobulk)")
    fig.suptitle(f"{DATASET}: donors with ≥{MIN_DA} DA nuclei", fontsize=10)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "donor_level_tyr_mao.png", dpi=150)
    plt.show()

## 8 · Save aggregate outputs (stay in the workspace)

In [ ]:
donor_tbl.to_csv(OUT_DIR / "donor_table.csv")
detect.to_csv(OUT_DIR / "detection_summary.csv", index=False)
if CELLTYPE_COL:
    xt.to_csv(OUT_DIR / "pipeline_label_vs_marker_call.csv")
pd.Series({"dataset": DATASET, "h5ad": str(h5ad_path), "region_col": REGION_COL, "donor_col": DONOR_COL,
           "dx_col": DX_COL, "celltype_col": CELLTYPE_COL, "sn_pattern": SN_PATTERN,
           "min_da": MIN_DA, "counts_layer": COUNTS_LAYER}).to_csv(OUT_DIR / "run_config.csv")
print("Written to", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()): print(" ", p.name)

## What to look at before going further

- **Section 4:** how many marker-defined DA nuclei per donor? If most donors have < 20, pool datasets
  (Lee + Sulzer, then Voet midbrain once curated) before any disease comparison.
- **Section 5:** TYR detection rate in DA nuclei. If it is near zero, check ambient/CellBender effects by rerunning
  on the raw CellRanger matrices (`preprocess/*.raw_feature_bc_matrix.h5`), since CellBender can strip very
  low-abundance transcripts. That decides whether TYR UMI-level analysis is feasible at all.
- **Section 6–7:** treat everything as a sanity check. The proper next step is a pseudobulk model
  (e.g. DESeq2/edgeR on donor sums) with age, sex, PMI and dataset as covariates, and SOX6/AGTR1 vs CALB1
  DA subtypes analysed separately.